# 01 — Dataset Collection and Assembly

> **Notebook status: documented reconstruction.** This notebook documents Dataset V2 from the project records and the recovered source manifest. It is not presented as the original executed V2 assembly notebook. The source image archives and full per-image V2 source manifest were not available in the uploaded application ZIP or recovered project files, so this notebook validates the source inventory and provides reusable checks, but it cannot rebuild the 1,250 image files in this repository alone.

## Goal

Explain how the balanced Dataset V2 was assembled from multiple archives, how the real and AI-generated labels were assigned, and how file-level provenance and deduplication checks fit into the pipeline.

## Source archive inventory

| Source archive | Role in Dataset V2 | Contribution recorded in project notes |
|---|---|---:|
| `real_car_dataset_v2_deduplicated.zip` | Existing real-car collection, including material from the project dataset and Hugging Face `Smily6820/car-images` | 625 real candidates |
| `Car_AI_Images_FGTS_Additions.zip` | Additional AI-generated car candidates sourced from FGTS | 474 AI candidates |
| `Car_Image_Authenticity_Dataset_v1.zip` | AI-generated images retained from the earlier V1 dataset | 151 AI candidates |
| **Total** | **Balanced Dataset V2** | **625 real + 625 AI = 1,250** |

The final 625/625 class balance is a project-record value. The archives themselves are intentionally not included in this source repository.

## FGTS manifest evidence

The recovered `collection_summary.json` records 474 extracted candidates, split evenly across three source-folder families: 158 LDM, 158 Stable Diffusion 1.4 and 158 ProGAN. It records seed 42 and says no invalid/duplicate candidates were skipped during that extraction step. This is the FGTS candidate extraction summary; it is not the complete post-merge Dataset V2 manifest.

The recovered AI-generation notebook for an earlier 100-image car pilot planned 34 Stable Diffusion v1.5, 33 SDXL Turbo and 33 Tiny-SD outputs. That is a separate historical pilot and is **not** counted as part of the 474 FGTS additions unless a later source manifest explicitly shows otherwise.

## Real-image sourcing evidence

Project notes identify the existing project real-car collection and the Hugging Face dataset [`Smily6820/car-images`](https://huggingface.co/datasets/Smily6820/car-images). A recovered script preserves a fixed selection of 51 Hugging Face candidate IDs and records the repository's declared CC BY 4.0 licence and upstream duplicate note. This script proves a 51-image selection workflow existed; it does not prove that all 625 real images came from Hugging Face.

## Assembly principles

1. Keep source archives unchanged and record their names and checksums.
2. Assign each selected image a class from source folder/manifest evidence; do not infer authenticity from appearance alone.
3. Preserve the archive name and original member path for each candidate.
4. Verify readability before the merge.
5. Detect exact duplicates using file SHA-256 and decoded-pixel hashes.
6. Use perceptual hashes to create a manual near-duplicate review queue; do not automatically delete solely because a perceptual hash is close.
7. Keep the classes balanced at 625 candidates each.

## Known provenance limitations

- The exact per-generator breakdown for the 151 V1 AI images was not recovered.
- The full real-image source breakdown within the 625 candidates was not recovered.
- The 51-image Hugging Face selection script is supporting evidence, not a complete source manifest.
- Labels and generator names were inherited from source folders/manifests; they were not independently verified for every image.


In [ ]:
from pathlib import Path
import hashlib, json, zipfile

PROJECT_ROOT = Path.cwd()
ARCHIVE_DIR = PROJECT_ROOT / "data" / "source_archives"  # Not shipped in this repository.
SOURCE_ARCHIVES = {
    "real": ARCHIVE_DIR / "real_car_dataset_v2_deduplicated.zip",
    "fgts_ai": ARCHIVE_DIR / "Car_AI_Images_FGTS_Additions.zip",
    "v1_ai": ARCHIVE_DIR / "Car_Image_Authenticity_Dataset_v1.zip",
}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tif", ".tiff", ".heic", ".heif"}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def inspect_archive(path: Path) -> dict:
    if not path.is_file():
        return {"path": str(path), "present": False, "image_members": None, "sha256": None}
    with zipfile.ZipFile(path) as archive:
        bad_member = archive.testzip()
        image_members = [n for n in archive.namelist() if Path(n).suffix.lower() in IMAGE_EXTENSIONS]
    return {
        "path": str(path), "present": True, "bytes": path.stat().st_size,
        "sha256": sha256_file(path), "image_members": len(image_members),
        "crc_check": "PASS" if bad_member is None else f"FAIL: {bad_member}",
    }

archive_inventory = [inspect_archive(p) for p in SOURCE_ARCHIVES.values()]
archive_inventory


## Recovered FGTS extraction summary

The JSON below is preserved at `documentation/source_evidence/fgts_collection_summary.json`. The values describe the FGTS extraction stage only.

In [ ]:
fgts_summary_path = PROJECT_ROOT / "documentation" / "source_evidence" / "fgts_collection_summary.json"
if fgts_summary_path.exists():
    fgts_summary = json.loads(fgts_summary_path.read_text(encoding="utf-8"))
else:
    # Also works when this notebook is run from its own folder.
    fgts_summary_path = Path("../documentation/source_evidence/fgts_collection_summary.json")
    fgts_summary = json.loads(fgts_summary_path.read_text(encoding="utf-8"))
fgts_summary


## Candidate manifest checks

A full Dataset V2 per-image manifest is not included. If the original manifest is available outside this repository, place a copy at `data/source_manifest.csv`. The code below checks that it has one row per candidate, contains 625 labels in each class, and has no duplicate file SHA-256 values. It does not guess the V1 archive's internal folder mapping.

In [ ]:
import pandas as pd

manifest_path = PROJECT_ROOT / "data" / "source_manifest.csv"
if manifest_path.exists():
    source_manifest = pd.read_csv(manifest_path)
    required = {"label", "source_archive", "source_member", "sha256"}
    missing = required - set(source_manifest.columns)
    if missing:
        raise ValueError(f"Source manifest is missing required columns: {sorted(missing)}")
    counts = source_manifest["label"].value_counts().to_dict()
    print("Candidate rows:", len(source_manifest))
    print("Counts by label:", counts)
    assert counts.get("real", 0) == 625
    assert counts.get("ai_generated", 0) == 625
    assert not source_manifest["sha256"].duplicated().any(), "Duplicate SHA-256 file hashes detected."
else:
    print("No per-image source manifest found. The recorded balance is 625 real + 625 AI-generated = 1,250.")
    print("Place the original manifest at data/source_manifest.csv to run the per-image checks.")


## Reusable decoded-pixel hash check

A file SHA-256 changes when a file is re-encoded. A decoded-pixel hash can find visually identical pixel arrays stored in different file containers. This helper is suitable for future audits when the candidate images are available.

In [ ]:
from PIL import Image, ImageOps
import hashlib

def decoded_pixel_sha256(image_path: Path) -> str:
    with Image.open(image_path) as image:
        rgb = ImageOps.exif_transpose(image).convert("RGB")
        payload = f"{rgb.width}x{rgb.height}:".encode() + rgb.tobytes()
    return hashlib.sha256(payload).hexdigest()

# Example: run on one available image, not on a missing dataset.


## Dataset V2 balance summary

| Class | Count |
|---|---:|
| Real | 625 |
| AI-generated | 625 |
| **Total** | **1,250** |

These are the documented candidate counts for Dataset V2, not a fresh count from image files in this repository. Raw datasets are excluded to keep this repository focused on code, model and academic evidence.